# 📏 The Evaluation — seven questions a skeptic would ask

In [`12_grand_finale.ipynb`](12_grand_finale.ipynb) the whole play ran, live: real chain,
real signatures, the real controller, one judgment call. That was an **existence proof** —
the system *can* run. A thesis committee, a code reviewer, a customer: none of them stops
there. They ask what it *costs*, how *fast* it is, how *cheatable* it is, and how close the
testbed is to the real deal. Answering those questions with numbers — each number carrying
its own honesty label — is an **evaluation**, and the way it is done here is a method you
can reuse on any system you ever build.

This chapter owns the *method*: the questions, the definitions fixed before measuring, the
harness that measured, and the dataset it left behind. The *numbers themselves* — reading
them, doubting them, concluding from them — belong to
[`14_results_and_conclusions.ipynb`](14_results_and_conclusions.ipynb).

**What you'll be able to do after this notebook**

- say what an evaluation adds over an existence proof, and name the skeptical question each
  of the seven experiments E1–E7 answers
- quote the two definitions fixed *before* any measurement: the two honest meanings of
  "enforced" in this lab, and the five simulation boundaries that cap every number in 14
- walk the harness ([`experiments.py`](../../src/e2e/experiments.py)) and explain why a
  measuring instrument must satisfy the same port as the thing it times (rule 7, third
  appearance)
- run experiment E7 — the predicate microbenchmark — live on your machine, and compare it
  to the committed dataset the honest way: orders of magnitude, not digits
- answer "where does that figure come from?" for any number in chapter 14 with a file path
  under `e2e/runs/eval/`

**You need:** notebooks 00–12. Infrastructure: **none** — E7 is pure Python (no chain, no
lab, no LLM), and every other experiment is *read* from the dataset committed in the repo,
not re-run. **Estimated time:** ~55 minutes.

## 1. "It ran once" is not an argument — the seven skeptical questions

What does the grand finale actually prove? That one lifecycle, on one afternoon, on one
machine, completed. A skeptic — your future self included — immediately asks more. And
here is the discipline this repo follows: **each experiment exists to answer exactly one
skeptical question.** Not "let's collect some metrics" — a named doubt, and the one
measurement that settles or bounds it.

| id | the skeptic asks | the harness answers with |
|---|---|---|
| **E1** | *"isn't it slow?"* | `exp_latency` — n=20 full lifecycles, every phase on its own stopwatch |
| **E2** | *"does the token actually govern the wire?"* | `exp_expiry` + the revocation leg inside every E1 lifecycle (plus `exp_revlag_sweep`, "E9", for the follow-up *"isn't that lag just your polling choice?"*) |
| **E3** | *"what does it cost?"* | gas receipts collected during E1 (the `_gas` helper), per transaction |
| **E4** | *"can it be cheated?"* | `exp_adversarial` — a dozen attacks, each attributed to the layer that rejected it |
| **E5** | *"is LLM judgment viable?"* | `exp_llm` — schema validity, accuracy against ground truth, tokens per call |
| **E6** | *"what does trustlessness cost over just calling the router?"* | `exp_baseline` — the same provisioning with no agents, no chain, no controller |
| **E7** | *"is the architecture itself slow?"* | `exp_predicate` — the authorization decision, isolated, in nanoseconds |

The next cell imports the harness and checks this table against the code — every claimed
function must actually exist. Look for the harness's own one-paragraph thesis at the top of
the output.

In [ ]:
import inspect
import json
import statistics
import tempfile
from pathlib import Path

import e2e.experiments as harness

ROOT = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / ".git").exists())
EVAL = ROOT / "e2e" / "runs" / "eval"                  # the committed dataset (evidence, in git)
SCRATCH = Path(tempfile.mkdtemp(prefix="course13_"))   # where OUR runs write — never EVAL
DATA_OK = (EVAL / "summary.json").exists()
if not DATA_OK:
    print("!! committed dataset missing — restore it:  git checkout -- e2e/runs/eval")

print(inspect.getdoc(harness).split("\n\n")[0], "\n")
ANSWERED_BY = {
    "E1 isn't it slow?": harness.exp_latency,
    "E2 does the token govern the wire?": harness.exp_expiry,
    "E3 what does it cost?": harness._gas,
    "E4 can it be cheated?": harness.exp_adversarial,
    "E5 is LLM judgment viable?": harness.exp_llm,
    "E6 what does trustlessness cost?": harness.exp_baseline,
    "E7 is the architecture itself slow?": harness.exp_predicate,
}
for question, fn in ANSWERED_BY.items():
    assert callable(fn)
    print(f"  {question:38} → {fn.__name__}")
print("\n✓ seven questions, seven pieces of code — the table is real")

**✏️ Your turn 1.1 — pin the question to the experiment**

Below are four skeptic's questions in scrambled order. Fill each `"E?"` with the right
experiment id (`"E1"`…`"E7"`), then un-comment the self-check. Success: the assert passes
silently — you can name the doubt each experiment settles without looking at the table.

In [ ]:
quiz = {
    "can it be cheated?": "E?",                                          # TODO
    "what does trustlessness cost over just calling the router?": "E?",  # TODO
    "is the architecture itself slow?": "E?",                            # TODO
    "does the token actually govern the wire?": "E?",                    # TODO
}
for question, eid in quiz.items():
    print(f"  {eid:3} ← {question}")

# un-comment once filled in:
# assert list(quiz.values()) == ["E4", "E6", "E7", "E2"]

<details><summary>✅ Solution 1.1 — peek only after trying</summary>

```python
quiz = {
    "can it be cheated?": "E4",
    "what does trustlessness cost over just calling the router?": "E6",
    "is the architecture itself slow?": "E7",
    "does the token actually govern the wire?": "E2",
}
assert list(quiz.values()) == ["E4", "E6", "E7", "E2"]
```

The subtle pair is E7 vs E1: E1 times the whole pipeline (chain, gNMI, signatures and all),
while E7 isolates the one piece this architecture *contributes* — the deterministic
predicate — so a slow chain can never be blamed on the design itself.
</details>

## 2. Fix the definitions before you look at the data

Here is how numbers get smuggled: measure first, then choose definitions that make the
measurements look good. The defense is boring and absolute — **write the definitions down
before running anything**. [`docs/09-evaluation.md`](../../../docs/09-evaluation.md) §2
does exactly that. It fixes two things: what the word **"enforced"** may honestly mean in
this lab, and which parts of the testbed are **real** versus **simulated**.

The first is the ADR-006 story from [`09_netctl_the_hands.ipynb`](09_netctl_the_hands.ipynb):
containerized SR Linux *accepts and stores* QoS config but does not *enforce* it in its
datapath. So "enforced" has two honest senses — **(1)** *config committed via gNMI Set and
read back from the running config* (what every number in the dataset means), and **(2)**
*datapath physics* — packets actually rate-limited (proven separately by the console's
iperf plateau, and never claimed by these numbers).

The next cell prints §2 verbatim from the committed report — read both definitions in
their fixed, pre-measurement form.

In [ ]:
report = (ROOT / "docs" / "09-evaluation.md").read_text()
sec2 = "## 2. " + report.split("## 2. ")[1].split("## 3. ")[0]
print(sec2)

assert "read back" in sec2 and "ADR-006" in sec2   # the two senses of "enforced"
assert "five things are simulated" in sec2          # the boundary list, pre-declared
assert "auto-mines" in sec2 and "Qwen3-4B" in sec2
print("✓ both definitions were fixed in writing before a single sample was taken")

The second half of §2 is the **five simulation boundaries** — the precise content of the
phrase "a system very close to the real deal". Read the two columns below together:
nothing in the left column was mocked (that is where the credibility comes from), and each
entry on the right caps one *specific class of claim* rather than vaguely weakening
everything. The lab is also small in ways §2 folds into B4: a two-host topology, no
adversarial network between the parties, lab-scale load only.

Every number chapter 14 shows you will carry exactly one of these labels. That is the
contract: **a number without its boundary is an overclaim.**

In [ ]:
REAL = [
    "a real Anvil EVM executing the real Settlement.sol (06)",
    "real EIP-712 / EIP-191 signatures, verified on-chain and in the controller (07)",
    "the real controller code path — predicate, state machine, auth (08)",
    "a real SR Linux config plane, written and read back over gNMI (09)",
    "the real deployed Qwen3-4B on Modal, for llm-mode runs (10)",
]
SIMULATED = {
    "B1 instant-finality chain": "Anvil auto-mines → every chain latency is a LOWER BOUND",
    "B2 in-process, co-located": "no A2A/HTTP hop in the timed path → transport-free lower bound",
    "B3 container datapath":     "config committed + read back ≠ packets shaped (ADR-006)",
    "B4 lab-scale load":         "n=20, sequential, one warm machine → typical cost, not tails/throughput",
    "B5 one LLM, one session":   "one model, one deployment → LLM claims don't generalize",
}
print("REAL (unmocked):")
for item in REAL:
    print("  ✓", item)
print("\nSIMULATED (each caps a claim):")
for boundary, cap in SIMULATED.items():
    print(f"  ✗ {boundary:27} → {cap}")
assert len(REAL) == len(SIMULATED) == 5

**✏️ Your turn 2.1 — tag the claim with its boundary**

Five claims that chapter 14 will make, one per boundary. Fill each `"B?"` with the
boundary id from the table above, then un-comment the check. Success: the assert passes —
you can attach the honesty label to a number before anyone asks you to.

In [ ]:
claims = {
    "settlement took 38 ms":                                 "B?",  # TODO
    "the 89 ms lifecycle contains zero network hops":        "B?",  # TODO
    "'enforced' means committed and read back, not shaped":  "B?",  # TODO
    "we quote medians and ranges, never tail latencies":     "B?",  # TODO
    "the decide slot scored 12/12 — on THIS model":          "B?",  # TODO
}
for claim, boundary in claims.items():
    print(f"  {boundary:3} ← {claim}")

# un-comment once tagged:
# assert list(claims.values()) == ["B1", "B2", "B3", "B4", "B5"]

<details><summary>✅ Solution 2.1 — peek only after trying</summary>

```python
claims = {
    "settlement took 38 ms":                                 "B1",
    "the 89 ms lifecycle contains zero network hops":        "B2",
    "'enforced' means committed and read back, not shaped":  "B3",
    "we quote medians and ranges, never tail latencies":     "B4",
    "the decide slot scored 12/12 — on THIS model":          "B5",
}
assert list(claims.values()) == ["B1", "B2", "B3", "B4", "B5"]
```

B1 makes 38 ms a *floor* (a real chain adds block time); B2 makes 89 ms a floor too
(transport is absent); B3 renames what "enforced" may claim; B4 forbids tail/throughput
talk; B5 confines every LLM statement to this one deployment.
</details>

## 3. The harness eats its own dog food

Now the instrument itself. There is a classic trap in benchmarking: to time a system you
*modify* it — add hooks, swap components, patch methods — and then your numbers describe
the modified system, not the one you ship. The harness avoids the trap with a move you have
now seen twice: in [`03_protocols_and_ports.ipynb`](03_protocols_and_ports.ipynb) the port
made mock and adapter interchangeable; in [`09_netctl_the_hands.ipynb`](09_netctl_the_hands.ipynb)
rule 7 forced the mock to pass the real adapter's contract tests. Third appearance: **the
measuring instrument satisfies the same port as the thing it times**, so the controller
cannot tell it is being watched.

The mechanism is a *delegation wrapper*: hold the real object, forward every call
untouched, and note the clock around the forward. A toy first — `Timed` puts a stopwatch on
`Snail` while neither `Snail` nor its caller changes at all:

In [ ]:
from time import perf_counter, sleep

class Snail:
    def work(self, ms):
        sleep(ms / 1000)
        return "done"

class Timed:
    def __init__(self, inner):
        self.inner = inner
        self.last = {}
    def work(self, *a, **k):                    # same method name = same shape
        t0 = perf_counter()
        result = self.inner.work(*a, **k)       # *a/**k pass any arguments through untouched
        self.last["work_s"] = perf_counter() - t0
        return result

timed_snail = Timed(Snail())
print(timed_snail.work(30), f"→ measured {timed_snail.last['work_s'] * 1000:.0f} ms")
assert timed_snail.work(30) == Snail().work(30)   # the wrapper changed nothing observable
assert 0.02 < timed_snail.last["work_s"] < 0.5

The real thing is `TimingProvisioner`, and the punchline is the `isinstance` check below:
the wrapper *and* the real `GnmiProvisioner` both satisfy `NetworkProvisioner` — the port
from 03. `ControllerService` receives the wrapper in its provisioner seat and runs
**unmodified production code**; the only trace is a `.last` dict of timings the harness
reads afterwards. That split is what lets the report separate "controller compute" from
"the gNMI Set" inside `activate()`.

In [ ]:
from a2a_interfaces import NetworkProvisioner
from netctl.provisioner import GnmiProvisioner

print(inspect.getsource(harness.TimingProvisioner))

timed = harness.TimingProvisioner(inner=None)   # inner is untouched until a call is forwarded
real = GnmiProvisioner({})                      # empty target map — construction is offline
for name, obj in [("TimingProvisioner", timed), ("GnmiProvisioner  ", real)]:
    ok = isinstance(obj, NetworkProvisioner)
    print(f"  isinstance({name}, NetworkProvisioner) → {ok}")
    assert ok
print("✓ the instrument and the instrumented walk through the same door (rule 7)")

**✏️ Your turn 3.1 — why not just monkeypatch?**

A tempting shortcut exists: `GnmiProvisioner.apply_bandwidth = timed_version` — reach into
the class at runtime and replace the method (that move is called *monkeypatching*). Write
your answer as a comment — what does the wrapper-at-the-port give you that the patch does
not? — then un-comment the self-check, which verifies the wrapper carries every port
method. Hint: think about *which system* your numbers would then describe, and who else
shares that class.

In [ ]:
# TODO: your answer as a comment first:
# why_wrap_not_patch = "..."

port_methods = {"apply_bandwidth", "apply_telemetry", "teardown", "health"}
wrapper_methods = {m for m in dir(harness.TimingProvisioner) if not m.startswith("_")}
print("the port needs   :", sorted(port_methods))
print("the wrapper offers:", sorted(wrapper_methods))

# un-comment once you've answered:
# assert port_methods <= wrapper_methods
# assert isinstance(harness.TimingProvisioner(None), NetworkProvisioner)

<details><summary>✅ Solution 3.1 — peek only after trying</summary>

```python
assert port_methods <= wrapper_methods
assert isinstance(harness.TimingProvisioner(None), NetworkProvisioner)
```

Monkeypatching mutates the class itself: *every* `GnmiProvisioner` in the process becomes
the mutant, the production code under test is no longer production code, and un-patching
reliably is hard. The wrapper leaves `GnmiProvisioner` byte-identical, is chosen visibly at
composition time (11's lesson), and the port guarantees the controller cannot tell — so the
numbers describe the real system plus a clock, not a mutant.
</details>

The harness's composition root — 11's word — is `Stack`: one disposable world per
experiment. A pinned Anvil, Ada/Bell/Mallory as `ChainClient`s, a `ChainReader`, the timed
provisioner, and the real `ControllerService`. Two things to look for below: the field list
reads like 12's cast list, and `watch()` arms **the production revocation watcher** — the
same `watch_revoked` polling thread you met in
[`07_chainmcp_the_signing_adapter.ipynb`](07_chainmcp_the_signing_adapter.ipynb) — not some
harness shortcut. When E2 reports a revocation lag, the mechanism under test is the real one.

In [ ]:
import dataclasses

print("Stack fields:", [f.name for f in dataclasses.fields(harness.Stack)])
print()
print(inspect.getsource(harness.Stack.watch))
assert "watch_revoked" in inspect.getsource(harness.Stack.watch)
print("✓ E2's revocation lag rides the real watcher thread — production mechanism, measured")

`run_lifecycle` is 12's play with a stopwatch on every act. The pattern is *bracket
timing* — `t0 = perf_counter()` … do the act … `phases[key] = perf_counter() - t0` —
repeated per phase. The regex below harvests every stopwatch key from the source; read them
as the acts of the finale (sign the offer → decide → settle → challenge → prove →
activate → …). Then the revocation leg in full: the revoke transaction is mined, and the
harness **polls the device** until the policer is gone — the time between those two moments
is E2's headline number, measured at the wire, not at the API.

In [ ]:
import re

src = inspect.getsource(harness.run_lifecycle)
phase_keys = list(dict.fromkeys(re.findall(r'phases\["(\w+)"\]', src)))
for key in phase_keys:
    print("  ⏱", key)
assert "revocation_lag_s" in phase_keys and "e2e_request_to_enforced_s" in phase_keys
print()
print(src[src.index("# 5. revoke"):])

"Evidence or it didn't happen" (rule 10) is code here too. Every experiment writes its
rows to a **JSONL** file — *JSON Lines*: one complete JSON object per line of a plain text
file. It is the humblest possible database: `cat`-able, `tail`-able, appendable without
rewriting anything, and a crash loses at most the one line being written. `_write` and
`_read` below are the entire storage engine (leading `_` means internal — we peek to
learn). Watch the toy round-trip: three dicts out, the same three back, byte-for-byte
inspectable in between.

In [ ]:
print(inspect.getsource(harness._write))
print(inspect.getsource(harness._read))

rows = [{"exp": "toy", "run": i, "ok": True} for i in range(3)]
harness._write(SCRATCH / "toy.jsonl", rows)
print((SCRATCH / "toy.jsonl").read_text(), end="")
assert harness._read(SCRATCH / "toy.jsonl") == rows
print("✓ round-trip exact — the file IS the evidence")

One more harness function, with a scar worth reading: `_stats` folds samples into
`median / mean / min / max` plus a *nearest-rank p95* — and its docstring documents a bug
that an **adversarial audit** caught before the report was written.
[`docs/evidence/M7.1.md`](../../../docs/evidence/M7.1.md) tells the story: a 4-agent review
panel attacked the first summary and forced five corrections. The star bug: with 20 sorted
samples, `values[int(0.95 * 20)]` is `values[19]` — the **maximum**, dressed up as a
percentile. Chapter 14 owns *reading* statistics honestly; take the method lesson here:
**audit your own numbers before you publish them.**

**✏️ Your turn 3.2 — resurrect the p95 bug**

The scaffold builds 20 fake samples and computes the "p95" both ways. Predict, as a
comment, which of the two equals the maximum — then un-comment the check. Success: the
buggy index IS the max (exactly why the audit flagged it), and the fixed one is not.

In [ ]:
import math

print(inspect.getsource(harness._stats))

vals = sorted(range(1, 21))                    # 20 samples: 1..20
buggy = vals[int(0.95 * len(vals))]            # the pre-audit indexing
fixed = vals[min(len(vals) - 1, math.ceil(0.95 * len(vals)) - 1)]   # nearest-rank
# TODO: your prediction as a comment — which one equals max(vals) == 20?
print(f"buggy p95 = {buggy}    fixed p95 = {fixed}    max = {max(vals)}")

# un-comment once you've predicted:
# assert buggy == max(vals) and fixed < max(vals)

<details><summary>✅ Solution 3.2 — peek only after trying</summary>

```python
assert buggy == max(vals) and fixed < max(vals)
```

`int(0.95 * 20) = 19` — the last index of a 20-element list, i.e. the sample maximum
wearing a percentile costume; nearest-rank (`ceil(0.95·n) − 1 = 18`) at least picks a real
in-between rank, which is why docs/09 still prefers to cite median + [min, max] at n=20.
</details>

## 4. Run E7 yourself — the predicate under a stopwatch

Why is E7 "the sharpest number" in the whole evaluation? Every other experiment times
*someone else's* machinery too — Anvil, gNMI, an LLM deployment. E7 isolates the one thing
this architecture actually *contributes*: the deterministic authorization decision,
`controller.domain.predicate` — the bouncer from
[`08_controller_the_bouncer.ipynb`](08_controller_the_bouncer.ipynb). And it is honestly
benchmarkable for a reason you can state: it is a **pure function** (rule 4 — no I/O, same
inputs → same answer, every single time). First, refresh your hands on it — `None` means
*allowed*, anything else is the **first failing** `ErrorCode`:

In [ ]:
from controller.domain import predicate
from a2a_interfaces import ErrorCode
from a2a_interfaces.fixtures import ADA, CANONICAL_ENTITLEMENT_VIEW as VIEW, WINDOW

mid = WINDOW.start + 60                       # 14:01 — inside the canonical window
stranger = "0x" + "b" * 40
print("in-window, Ada asks      →", predicate(VIEW, ADA, ADA, mid, set()), "  ← None = ALLOWED")
print("a stranger asks          →", predicate(VIEW, ADA, stranger, mid, set()))
print("Ada, before the window   →", predicate(VIEW, ADA, ADA, WINDOW.start - 10, set()))
print("Ada, after the window    →", predicate(VIEW, ADA, ADA, WINDOW.end + 10, set()))
print("Ada, session already on  →", predicate(VIEW, ADA, ADA, mid, {VIEW.id}))
assert predicate(VIEW, ADA, ADA, mid, set()) is None
assert predicate(VIEW, ADA, stranger, mid, set()) is ErrorCode.E_NOT_OWNER
assert predicate(VIEW, ADA, ADA, mid, {VIEW.id}) is ErrorCode.E_CONFLICT

New construct: **`timeit`** — the standard library's microbenchmark tool. You hand it a
zero-argument callable and a loop count; it runs the callable that many times against a
high-resolution clock and returns total seconds. Divide by the loops → seconds per call;
multiply by 10⁹ → **nanoseconds**, billionths of a second — the natural scale of a single
Python function call. Toy first. Note two things: the measured cost *includes* the
lambda-call overhead (fine, as long as you say so), and the second line shows the
harness's `lambda a=args: ...` trick — a default argument freezes the inputs at definition
time, so the timed loop pays no global-variable lookups.

In [ ]:
import timeit

loops = 100_000
secs = timeit.timeit(lambda: 2 + 2, number=loops)
print(f"2 + 2 → {secs / loops * 1e9:6.0f} ns/call   (includes the lambda-call overhead)")

args = ("hello", "world")
join = timeit.timeit(lambda a=args: " ".join(a), number=loops)   # a=args freezes the inputs
print(f"join  → {join / loops * 1e9:6.0f} ns/call   ← the default-arg trick the harness uses")
assert secs > 0 and join > 0

`exp_predicate` runs exactly that for **all seven outcomes**, 200,000 calls each — and,
like every experiment, it *writes its rows to disk*. One safety rule before you run it: the
CLI's default `--out` is `e2e/runs/eval/`, the **committed evidence**. A casual rerun
pointed there would overwrite the dataset that docs/09 and chapter 14 cite. We pass
`SCRATCH` instead — the throwaway temp directory from §1.

**✏️ Your turn 4.1 — predict your machine, then run**

The committed run measured **85.7 ns** for the allow path and **51–83 ns** for the denials
(the author's machine). Write a prediction for *your* machine as a comment — faster?
slower? by how much? — then run the cell: it executes the real E7, 1.4 million predicate
calls, in well under a second. Un-comment the self-check afterwards.

In [ ]:
# TODO: your prediction first, as a comment — allow-path ns on YOUR machine:
# my_prediction_ns = ...

rows = harness.exp_predicate(SCRATCH)          # writes SCRATCH/predicate.jsonl — never EVAL!
mine = {r["outcome"]: r["ns_per_call"] for r in rows}
print(f"\nyour allow path: {mine['allow']:.0f} ns    (the committed run: 85.7 ns)")

# un-comment once you've predicted:
# assert all(ns < 1_000 for ns in mine.values()), "every outcome should stay under a microsecond"

<details><summary>✅ Solution 4.1 — peek only after trying</summary>

```python
assert all(ns < 1_000 for ns in mine.values())
```

Whatever your CPU, all seven outcomes land in the tens-to-hundreds of nanoseconds — under
a microsecond — because the predicate is six comparisons over in-memory fields, no I/O
(rule 4). Early denials (`E_NOT_OWNER`) exit after one check and run cheapest; late ones
(`E_SCOPE`, `E_CONFLICT`) walk nearly the whole ladder and can measure within noise of
*allow* — the cost ordering mirrors the check ordering.
</details>

Your numbers do not read 85.7. So — is the dataset wrong, or are you? Neither, and this
is the comparison discipline: **nanosecond microbenchmarks do not replicate across machines
in their digits; they replicate in their order of magnitude.** CPU model, clock speed,
cache state, Python version — all shift the digits. What must survive the trip to your
machine is the *shape* of the claim: "tens of nanoseconds — three to four orders of
magnitude below the LLM slots and the chain". The cell compares your live run against the
committed one and asserts exactly that: a ratio within 10×, never equality.

In [ ]:
committed = {r["outcome"]: r["ns_per_call"] for r in harness._read(EVAL / "predicate.jsonl")}
if committed:
    print(f"{'outcome':14}{'yours':>9}{'committed':>11}{'ratio':>9}")
    for outcome, ns in mine.items():
        ratio = ns / committed[outcome]
        print(f"{outcome:14}{ns:>8.0f} {committed[outcome]:>10.1f} {ratio:>8.2f}×")
        assert 0.1 < ratio < 10, "cross-machine, the honest claim is SAME ORDER OF MAGNITUDE"
    print("\n✓ digits differ, magnitude agrees — that IS replication, for a microbenchmark")
else:
    print("skipped: committed predicate.jsonl missing — git checkout -- e2e/runs/eval")

**✏️ Your turn 4.2 — two sins, one verdict**

Build a view that is *both* revoked *and* past its window, then ask the predicate. Which
`ErrorCode` wins? The scaffold currently asks in-window (so you will see `E_REVOKED`);
move `when` past `WINDOW.end`, observe the flip, and un-comment the check. The answer is
08's check-order contract — *who → not-yet → expired → revoked → scope → conflict* — and
that fixed order is precisely the determinism that made E7's `timeit` honest.

In [ ]:
doomed = VIEW.model_copy(update={"revoked": True})    # revoked AND (soon) expired
when = WINDOW.start + 60                              # TODO: move this past WINDOW.end
verdict = predicate(doomed, ADA, ADA, when, set())
print("verdict:", verdict)

# un-comment once you've moved `when` past the window:
# assert verdict is ErrorCode.E_EXPIRED, "expired outranks revoked — the check order is a contract"

<details><summary>✅ Solution 4.2 — peek only after trying</summary>

```python
when = WINDOW.end + 10
verdict = predicate(doomed, ADA, ADA, when, set())
assert verdict is ErrorCode.E_EXPIRED
```

The expiry check sits before the revocation check, so the earliest sin answers — same
inputs, same verdict, forever. A predicate whose answer depended on evaluation order
whims could not be timed per-outcome at all.
</details>

## 5. The dataset: where every number in 14 lives

Chapter 14 will show you medians, gas figures, an adversarial matrix, LLM accuracy. Every
one of those figures derives from **eight files in one directory**, committed to git as
evidence (rule 10). Nothing arrives from nowhere: seven JSONL files — one per experiment,
raw rows — plus `summary.json`, the folded view that `summarize()` computes from them.
First, the inventory:

In [ ]:
if DATA_OK:
    expected = {"latency", "expiry", "baseline", "adversarial", "llm", "predicate", "revlag_sweep"}
    print(f"{'file':22}{'bytes':>8}   rows")
    for p in sorted(EVAL.iterdir()):
        n = len(harness._read(p)) if p.suffix == ".jsonl" else "—"
        print(f"{p.name:22}{p.stat().st_size:>8}   {n}")
    assert {p.stem for p in EVAL.glob("*.jsonl")} == expected
    print("\n✓ seven experiments, seven row files, one summary — the whole evidence base")
else:
    print("skipped: dataset not present — git checkout -- e2e/runs/eval")

Open one **latency** row — a single grand-finale performance, flattened to numbers.
`phases` is the stopwatch dict you harvested from `run_lifecycle` in §3, in seconds per
act; `gas` is E3 riding along (you met gas in
[`06_blockchain_from_zero.ipynb`](06_blockchain_from_zero.ipynb) — here it is simply three
receipts per lifecycle); `service` and `mode` say which of the four campaign arms
(det/llm × bandwidth/telemetry) this row belongs to.

In [ ]:
if DATA_OK:
    rec = harness._read(EVAL / "latency.jsonl")[0]
    print(json.dumps(rec, indent=1))
    assert rec["ok"] and "phases" in rec and "gas" in rec
    e2e_ms = rec["phases"]["e2e_request_to_enforced_s"] * 1000
    print(f"\n→ this run: request → enforced in {e2e_ms:.0f} ms · fulfill burned {rec['gas']['fulfill']:,} gas")
else:
    print("skipped: dataset not present")

Now one row from each of the other six files — enough to recognize each shape when 14
aggregates them. Notice how self-describing every row is: `exp` names the experiment, and
the payload keys say exactly what was measured (`ns_per_call`, `tick_to_deconfig_s`,
`attack`/`layer`, `slot`/`correct`, `poll_s`…).

In [ ]:
if DATA_OK:
    for name in ("predicate", "expiry", "baseline", "adversarial", "llm", "revlag_sweep"):
        first = harness._read(EVAL / f"{name}.jsonl")[0]
        shown = {k: first[k] for k in list(first)[:5]}
        print(f"{name:14} → {shown}")
else:
    print("skipped: dataset not present")

**✏️ Your turn 5.1 — find the fulfill-gas record**

Chapter 14 will claim a bandwidth `fulfill` costs ~268k gas. Which JSONL file holds the raw
numbers behind that claim? The scaffold guesses wrong on purpose — re-point `my_guess` at
the right file, then un-comment the check. Success: the first record carries a `gas` dict
with a `"fulfill"` key. (Hint: §1's table — E3 has no file of its own.)

In [ ]:
my_guess = EVAL / "expiry.jsonl"     # TODO: wrong on purpose — which file really carries gas?
rows_51 = harness._read(my_guess)
print(f"{my_guess.name}: first record keys → {list(rows_51[0]) if rows_51 else '(no data)'}")

# un-comment once you've re-pointed my_guess:
# assert rows_51 and "gas" in rows_51[0] and "fulfill" in rows_51[0]["gas"]

<details><summary>✅ Solution 5.1 — peek only after trying</summary>

```python
my_guess = EVAL / "latency.jsonl"
rows_51 = harness._read(my_guess)
assert rows_51 and "gas" in rows_51[0] and "fulfill" in rows_51[0]["gas"]
```

E3 is measured *inside* E1 — every lifecycle already pays the transactions, so `_gas` just
reads the receipts and stores them on the latency row; `summary.json` later folds them per
service type (fulfill is bimodal: telemetry offers carry bigger ABI params).
</details>

The eighth file, `summary.json`, is `summarize()`'s fold of all seven JSONLs: median and
spread per phase, gas per service type, the adversarial rows verbatim, LLM accuracy split
by boundary cases. It is the file chapter 14 reads most — but it is *derived*: delete it,
run `summarize()`, and it comes back, because the JSONLs are the ground truth. Note also
`watch_poll_s` at the top level: the watcher's poll interval is published *next to* the
revocation numbers it shapes — a boundary label living inside the dataset itself.

In [ ]:
if DATA_OK:
    S = json.loads((EVAL / "summary.json").read_text())
    print("top-level keys:", list(S))
    print("\ncampaign      :", S["runs_ok"], "/", S["runs_total"], "lifecycles ok ·",
          len(S["failures"]), "failures")
    print("predicate (ns):", S["predicate"])
    print("watcher poll  :", S["watch_poll_s"], "s  ← published WITH the lag it shapes")
    assert set(S["predicate"]) == {"allow", "E_NOT_OWNER", "E_NOT_STARTED", "E_EXPIRED",
                                   "E_REVOKED", "E_SCOPE", "E_CONFLICT"}
else:
    print("skipped: dataset not present")

The takeaway skill of this section: **every figure has an address.** Here is the
provenance map for the headline numbers 14 will defend. If you can produce the right-hand
side from memory for any figure you meet there, this notebook did its job.

In [ ]:
PROVENANCE = {
    "89 ms det lifecycle":       "latency.jsonl → phases.e2e_request_to_enforced_s (mode=det)",
    "3.3 s llm lifecycle":       "latency.jsonl → same key, mode=llm",
    "~86 ns predicate allow":    "predicate.jsonl → ns_per_call (outcome=allow)",
    "464 ms revocation lag":     "latency.jsonl → phases.revocation_lag_s (+ summary.watch_poll_s)",
    "73 ms expiry teardown":     "expiry.jsonl → tick_to_deconfig_s",
    "268k / 447k gas fulfill":   "latency.jsonl → gas.fulfill, folded per service in summary.json",
    "12/12 attacks rejected":    "adversarial.jsonl → rejected / layer / code per attack",
    "12/12 LLM decide accuracy": "llm.jsonl → correct (slot=decide)",
    "+69 ms trustlessness":      "latency.jsonl e2e median MINUS baseline.jsonl apply_s median",
    "lag ≈ poll curve":          "revlag_sweep.jsonl → revocation_lag_s per poll_s",
}
for claim, addr in PROVENANCE.items():
    fname = addr.split(" ")[0]
    have = (EVAL / fname).exists()
    print(f"  {'✓' if have else '✗'} {claim:27} ← {addr}")
    assert have or not DATA_OK
print("\n✓ ten headline figures, ten file addresses — no number arrives from nowhere")

**✏️ Your turn 5.2 — a five-line JSONL reader**

Prove the dataset needs no special tooling: compute the **median revocation lag** across
all successful latency rows, straight from the file. The scaffold loads the rows; you
extract `phases["revocation_lag_s"]` where present, take `statistics.median`, and
un-comment the check. Success: a number around half a second — the watcher's 0.5 s poll
shining through (which 14 will read properly, next to E9's sweep).

In [ ]:
lat_rows = harness._read(EVAL / "latency.jsonl")
lags = []   # TODO: [r["phases"]["revocation_lag_s"] for r in lat_rows if r["ok"] and "revocation_lag_s" in r["phases"]]
print(f"{len(lat_rows)} rows loaded · {len(lags)} lags extracted")
# median_lag = statistics.median(lags)
# print(f"median revocation lag: {median_lag * 1000:.0f} ms")

# un-comment once computed:
# assert 0.2 < median_lag < 0.7, "roughly half a 0.5 s poll of waiting + the actuation floor"

<details><summary>✅ Solution 5.2 — peek only after trying</summary>

```python
lags = [r["phases"]["revocation_lag_s"] for r in lat_rows
        if r["ok"] and "revocation_lag_s" in r["phases"]]
median_lag = statistics.median(lags)
assert 0.2 < median_lag < 0.7
```

Five lines, standard library only — that is the point of JSONL as an evidence format: any
reader, any language, no schema registry, and `summary.json` can always be re-derived.
</details>

## 6. Reproducing the campaign — the honest infra ladder

An evaluation you cannot re-run is an anecdote with charts. So the repo commits both the
dataset (which you just read) *and* the recipes that regenerate it. The entry point is one
`just` recipe — you met `just` in [`00_orientation.ipynb`](00_orientation.ipynb) as the
repo's command shortcut file. Read it below: the full campaign, then the lab-free E7, then
a re-render of the figures notebook.

In [ ]:
justfile = (ROOT / "Justfile").read_text()
recipe = justfile[justfile.index("eval n="):].split("\n\n")[0]
print(recipe)
assert "--exp predicate" in recipe and "e2e.experiments" in recipe
print("\n✓ `just eval` = campaign + E7 + figures, one command")

Under the recipe sits a plain **argparse** CLI (`argparse` is the standard library's
command-line parser: it turns `--exp predicate --n 5` typed in a terminal into
`args.exp == "predicate"` and `args.n == 5` in Python). Three flags select what runs:
`--exp` picks the experiment (or `all`), `--n` the sample count, `--mode det|llm|both`
whether the judgment slots call the live LLM. Look at the `choices=` list in the source —
it is §1's table again, this time as code — and at `--out`'s computed default, the exact
clobber hazard §4 warned you about.

In [ ]:
src = inspect.getsource(harness.main)
print(src[: src.index("    if args.exp")])
assert '"--exp"' in src and '"--mode"' in src and '"--n"' in src

Not every rung of the ladder is equally reachable, and the harness is honest about that
rather than quietly degrading: every lab-bound experiment calls `_require_lab()` first,
which *refuses* — naming the exact command you are missing — instead of substituting a
mock (the same cast-list ethic as 12). The cell below is happy either way: headless it
shows the refusal; if you actually have the containerlab running, it prints the lab's
address instead.

In [ ]:
print(inspect.getsource(harness._require_lab))
try:
    ip = harness._require_lab()
    print(f"lab is UP at {ip} — the lab-bound experiments would really run on this machine")
except SystemExit as refusal:
    print("SystemExit →", refusal)
    print("✓ an honest refusal that names the fix — never a silent mock")

**✏️ Your turn 6.1 — read the ladder off the source**

Which experiments need the router lab, and which don't? Don't guess — *scan the code*: an
experiment is lab-bound if and only if its source calls `_require_lab`. Write your
prediction first (there are seven `exp_*` functions — how many are lab-free, and which?),
run, then un-comment the check. Careful with the trap: one experiment is lab-**free** but
not infra-free.

In [ ]:
# TODO: your prediction first, as a comment — which exp_* functions do NOT call _require_lab?
# my_prediction = {"exp_...", "exp_..."}

exps = {n: getattr(harness, n) for n in dir(harness) if n.startswith("exp_")}
lab_free = {n for n, f in exps.items() if "_require_lab" not in inspect.getsource(f)}
for n in sorted(exps):
    print(f"  {n:20} {'lab-free' if n in lab_free else 'needs containerlab + anvil'}")

# un-comment once predicted:
# assert lab_free == {"exp_predicate", "exp_llm"}

<details><summary>✅ Solution 6.1 — peek only after trying</summary>

```python
assert lab_free == {"exp_predicate", "exp_llm"}
```

`exp_llm` is the trap: it needs no router — but it needs the deployed Modal endpoint
(`LLM_BASE_URL` / `LLM_MODEL` / `LLM_API_KEY`, ADR-001), and headless with a non-localhost
URL its readiness probe retries for up to ~8 minutes before giving up — never call it
casually. `exp_predicate` is the only truly zero-infrastructure rung, which is exactly why
this notebook could run it live.
</details>

The full ladder as terminal recipes — each rung adds exactly one piece of infrastructure
([`docs/09-evaluation.md`](../../../docs/09-evaluation.md) §13 is the canonical copy):

```sh
# rung 0 — no infrastructure at all (you just ran this live):
uv run python -m e2e.experiments --exp predicate

# rung 1 — Anvil + the SR Linux lab (E1, E2, E3, E4, E6, E9):
containerlab deploy -t netlab/topology.clab.yml
uv run python -m e2e.experiments --exp latency --n 5 --mode det   # quick smoke
uv run python -m e2e.experiments --exp all --n 20                 # the campaign

# rung 2 — plus the deployed LLM endpoint (E5, and --mode llm):
set -a && source .env && set +a
uv run python -m e2e.experiments --exp llm

# or all of it, figures re-rendered afterwards:
just eval
```

One last repeat of the safety rule, because it bites: `--out` defaults to
`e2e/runs/eval/` — the committed evidence. Reproducing for comparison? Pass
`--out /tmp/myeval` and diff against the committed files instead of overwriting them.

## What you learned (and where it goes)

| You did | The concept | Where it goes next |
|---|---|---|
| mapped E1–E7 to skeptic's questions | an evaluation = named doubts, one measurement each | 14 answers each question with its measured number |
| printed docs/09 §2 verbatim | definitions fixed BEFORE data: two senses of "enforced", five simulation boundaries | every figure in 14 carries one boundary label |
| `isinstance`'d TimingProvisioner against the port | rule 7, third appearance: the instrument passes the same contract as the thing it times | any benchmark you ever write over a ports-and-adapters system |
| round-tripped `_write`/`_read` | JSONL: evidence as append-only, cat-able rows (rule 10) | 14 reads these exact files |
| resurrected the p95 bug | audit your numbers before publishing (M7.1's panel) | 14's "median + range, no p95 at n=20" convention |
| ran `exp_predicate` live into SCRATCH | pure functions are honestly benchmarkable (rule 4) | 14's "the architecture's own cost is ~90 ns" claim |
| compared your ns to the committed ns | cross-machine discipline: orders of magnitude, not digits | how to read EVERY number in 14 |
| walked the provenance map + infra ladder | reproducibility as recipes: `just eval`, `--exp`, `--out` | your own re-run, if you bring the lab |

## Experiments to try

Predict first, then run:

- **Loop-count sensitivity:** re-time the allow case with `number=1_000` instead of
  200,000, five times in a row. Predict: will the five results agree in digits, or only in
  magnitude? (Then you'll know why `exp_predicate` uses 200k.)
- **The partial summary:** run `harness.summarize(SCRATCH)` — your scratch dir holds only
  `predicate.jsonl` and `toy.jsonl`. Predict which top-level keys the resulting
  `summary.json` will and won't contain, then read it back and check.
- **Controller-compute sliver:** across the det latency rows, compute
  `activate_s − gnmi_apply_s` per row. Predict its magnitude relative to the 21 ms gNMI
  write before you look — this difference is a bar 14 will draw.
- **Deliberate breakage:** from a terminal, run
  `uv run python -m e2e.experiments --exp latency --n 1` with no lab. Predict the exact
  wording of the refusal before you read it.

## Check yourself

Answer out loud before moving on:

1. A colleague says "the demo worked — ship the paper." Name the seven questions they have
   not answered, and the experiment that answers each.
2. What are the two honest meanings of "enforced" in this lab? Which one does every number
   in the dataset use, and what proves the other one?
3. Name the five simulation boundaries and, for each, one claim it caps.
4. Why does `TimingProvisioner` satisfy `NetworkProvisioner` instead of monkeypatching
   `GnmiProvisioner` — and what would the numbers describe if it didn't?
5. Your predicate measured 120 ns where the dataset says 86 ns. Is the dataset wrong? What
   claim survives both machines?

**Where this goes next:**
[`14_results_and_conclusions.ipynb`](14_results_and_conclusions.ipynb) — the numbers
themselves, read honestly: medians with their spreads, every figure with its boundary,
threats to validity, and the feasibility verdict this whole course has been building toward.

**Deeper dives:** [`evaluation_explore.ipynb`](../evaluation_explore.ipynb) (the compact
figures notebook that `just eval` re-renders) ·
[`docs/09-evaluation.md`](../../../docs/09-evaluation.md) (the written report, all caveats) ·
[`docs/evidence/M7.1.md`](../../../docs/evidence/M7.1.md) (the audit story: what a 4-agent
review panel forced the harness to fix) ·
[`e2e/src/e2e/experiments.py`](../../src/e2e/experiments.py) (the harness, whole).